# DistilBERT Fake Review Detection — Yelp — Supervised Contrastive Learning

**Model:** `distilbert-base-uncased` (fine-tuned with joint supervised-contrastive + cross-entropy loss)

**Method:** Single-stage joint training (Gunel et al., ICLR 2021 — *Supervised Contrastive Learning for Pre-trained Language Model Fine-Tuning*)

The encoder is trained with **both** losses at every step:

```
loss = (1 − λ)·CrossEntropy  +  λ·SupCon
```

- **CrossEntropy** drives the discriminative (fake vs genuine) signal and keeps the encoder task-relevant.
- **SupCon** (Khosla et al., 2020) pulls same-class CLS embeddings together and pushes different-class apart on the unit sphere, regularising the embedding geometry.

**Why joint, not staged:** A separate SupCon-only pretraining stage trains the encoder on the contrastive objective *alone*. On Yelp's behavioural labels (no text-level signal) this points the 67M pretrained parameters in a direction useless for classification — the earlier staged run dropped to 62% val macro F1 (vs 69% baseline). Joint training never lets SupCon move the encoder by itself: CE always co-supervises, so the contrastive term is a pure regulariser and cannot destroy the pretrained representations.

**Performance levers (tuned for maximum macro F1):**
- **`MAX_LEN = 256` (not 128):** ~44% of Yelp reviews exceed 128 tokens; at 128 nearly half of every review was truncated. 256 cuts truncation to ~17% and is the single biggest accuracy lever. *(The baseline and focal DistilBERT notebooks are matched at 256 too, so the three-way ablation is strictly comparable.)*
- **Layer-wise LR decay (LLRD):** top transformer layers adapt fast, lower layers stay near pretrained — standard recipe for best transformer fine-tuning.
- **Cosine schedule** with warmup; early stopping on val macro F1.
- **τ = 0.3, λ = 0.1:** Gunel et al.'s tuned optimum for text.

**Training data:** Full unbalanced Yelp dataset from `yelp_split_imbalanced.csv` (~13% fake), trained on class-balanced batches (50/50 per step) so SupCon always has positive pairs for both classes and CE matches the 50/50 balanced test set.

**Architecture:**
- Encoder: DistilBERT → CLS token → **768-dim**
- Projection head (SupCon only, discarded at inference): 768 → Dense(256, relu) → Dense(128) → L2-normalize → **128-dim unit sphere**
- Classifier head (used at inference): pre_classifier(768→768, relu) → Dropout(0.1) → Dense(768→2)

Both heads share a single encoder forward pass per step.

---
### Instructions
1. Go to **Runtime → Change runtime type → A100 GPU** before running.
2. Run `prepare_yelp_split_imbalanced.ipynb` first to generate `yelp_split_imbalanced.csv`.
3. Update file paths in **cell 3** if needed.
4. Run all cells in order.
5. Download `yelp_fake_distilbert_contrastive_onnx.zip`, extract it, and place the folder at `data/yelp_fake_distilbert_contrastive_onnx/`.

In [ ]:
import torch
print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))

In [ ]:
!pip install -q transformers accelerate optimum[onnxruntime]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── UPDATE THESE PATHS if your files are in a different location ──
SPLIT_CSV    = '/content/drive/MyDrive/yelp_split_imbalanced.csv'
BALANCED_CSV = '/content/drive/MyDrive/yelp_split.csv'

In [ ]:
import re, os, json, random, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import (
    DistilBertTokenizerFast,
    DistilBertModel,
    DistilBertForSequenceClassification,
    get_cosine_schedule_with_warmup,
)
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score, classification_report, precision_score, recall_score
from optimum.onnxruntime import ORTModelForSequenceClassification

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

In [ ]:
# ── Tokenisation ─────────────────────────────────────────────────
# 256, not 128: ~44% of Yelp reviews exceed 128 tokens (mean ≈153, median ≈113).
# At 128 nearly half of every review was truncated before the model saw it.
# 256 cuts truncation to ~17% and is the single biggest accuracy lever here.
MAX_LEN    = 256
MODEL_NAME = 'distilbert-base-uncased'

# ── Projection head (SupCon geometry only; discarded at inference) ─
PROJ_HIDDEN = 256   # 768 → 256
PROJ_OUT    = 128   # 256 → 128  (L2 normalised)

# ── Joint Supervised-Contrastive + Cross-Entropy fine-tuning ──────
# Single stage:  loss = (1 - λ)·CE  +  λ·SupCon   (Gunel et al., ICLR 2021)
#   - CE keeps the encoder task-relevant (cannot collapse on weak labels)
#   - SupCon regularises the embedding geometry
#   - SupCon never trains the encoder ALONE → cannot destroy pretrained weights
SUPCON_TEMP   = 0.3    # Gunel et al. found τ≈0.3 optimal for text fine-tuning
SUPCON_LAMBDA = 0.1    # contrastive weight; CE stays dominant on weak-signal Yelp labels
BATCH_SIZE    = 64     # 32 genuine + 32 fake — fits seq-256 on one A100; 63 SupCon negatives/anchor

# ── Optimisation ──────────────────────────────────────────────────
EPOCHS       = 7
LR           = 2e-5    # base (top) LR; lower layers decayed via LLRD below
LLRD_DECAY   = 0.9     # layer-wise LR decay — top layers adapt fast, bottom layers stay near pretrained
WARMUP_RATIO = 0.1
PATIENCE     = 3       # early stopping on val macro F1

In [ ]:
def supcon_loss(embeddings, labels, temperature=0.3):
    """
    Supervised Contrastive Loss (Khosla et al., 2020).
    Used here as an auxiliary term alongside cross-entropy (Gunel et al., 2021).
    embeddings : (N, D) float32 — L2-normalised
    labels     : (N,)  long
    """
    N      = embeddings.shape[0]
    device = embeddings.device

    sim     = torch.matmul(embeddings, embeddings.T) / temperature  # (N, N)
    sim_max = sim.max(dim=1, keepdim=True).values.detach()
    sim     = sim - sim_max

    mask_self = torch.eye(N, device=device).bool()
    mask_pos  = (labels.unsqueeze(0) == labels.unsqueeze(1)) & ~mask_self

    exp_sim   = torch.exp(sim) * (~mask_self).float()
    log_denom = torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-9)
    log_prob  = sim - log_denom

    n_pos    = mask_pos.float().sum(dim=1)
    mean_lp  = (mask_pos.float() * log_prob).sum(dim=1) / (n_pos + 1e-9)
    valid    = n_pos > 0
    loss     = -mean_lp[valid].mean() if valid.any() else sim.new_tensor(0.0)
    return loss

In [ ]:
def normalize_text(t):
    return re.sub(r'\s+', ' ', t.strip().lower())

print('Loading yelp_split_imbalanced.csv ...')
df_split = pd.read_csv(SPLIT_CSV)
df_split['text_norm'] = df_split['text'].apply(normalize_text)

X_train = df_split[df_split['split'] == 'train']['text'].tolist()
y_train = df_split[df_split['split'] == 'train']['label'].tolist()
X_val   = df_split[df_split['split'] == 'val']['text'].tolist()
y_val   = df_split[df_split['split'] == 'val']['label'].tolist()
X_test  = df_split[df_split['split'] == 'test']['text'].tolist()
y_test  = df_split[df_split['split'] == 'test']['label'].tolist()

print('\nSplit counts:')
for name, X, y in [('Train', X_train, y_train), ('Val', X_val, y_val), ('Test', X_test, y_test)]:
    n_g = sum(1 for v in y if v == 0)
    n_f = sum(1 for v in y if v == 1)
    print(f'  {name:<6}: {len(X):>8,}  genuine={n_g:,}  fake={n_f:,}  fake%={n_f/len(y):.1%}')

# ── Leakage check ────────────────────────────────────────────────
# Balanced test texts must not appear in the imbalanced train OR val splits.
print('\nLeakage check ...')
bal_df = pd.read_csv(BALANCED_CSV)
bal_df['text_norm'] = bal_df['text'].apply(normalize_text)

pinned_test = set(bal_df[bal_df['split'] == 'test']['text_norm'])
train_norms = set(df_split[df_split['split'] == 'train']['text_norm'])
val_norms   = set(df_split[df_split['split'] == 'val']['text_norm'])

leaked_train = pinned_test & train_norms
leaked_val   = pinned_test & val_norms

print(f'  Balanced test texts : {len(pinned_test):,}')
print(f'  Overlap with train  : {len(leaked_train):,}')
print(f'  Overlap with val    : {len(leaked_val):,}')

if leaked_train:
    raise ValueError(
        f'LEAKAGE: {len(leaked_train):,} balanced test texts found in training set. '
        'Regenerate yelp_split_imbalanced.csv.'
    )
if leaked_val:
    raise ValueError(
        f'LEAKAGE: {len(leaked_val):,} balanced test texts found in validation set. '
        'Regenerate yelp_split_imbalanced.csv.'
    )
print('  PASS — No leakage.')

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts, self.labels, self.tokenizer, self.max_len = texts, labels, tokenizer, max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }


class BalancedBatchSampler(Sampler):
    """Yields batches with exactly batch_size//2 genuine and batch_size//2 fake."""
    def __init__(self, labels, batch_size):
        self.labels     = np.array(labels)
        self.batch_size = batch_size
        self.idx_g      = np.where(self.labels == 0)[0]
        self.idx_f      = np.where(self.labels == 1)[0]
        self.half       = batch_size // 2
        self.n_batches  = len(self.labels) // batch_size
    def __iter__(self):
        for _ in range(self.n_batches):
            g   = np.random.choice(self.idx_g, self.half, replace=len(self.idx_g) < self.half)
            f   = np.random.choice(self.idx_f, self.half, replace=len(self.idx_f) < self.half)
            idx = np.concatenate([g, f])
            np.random.shuffle(idx)
            yield idx.tolist()
    def __len__(self):
        return self.n_batches


print(f'Loading tokenizer: {MODEL_NAME} ...')
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

train_dataset = ReviewDataset(X_train, y_train, tokenizer, MAX_LEN)
val_dataset   = ReviewDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_dataset  = ReviewDataset(X_test,  y_test,  tokenizer, MAX_LEN)

# Class-balanced batches (64 genuine + 64 fake). Required for joint training:
#   - SupCon needs positive pairs for BOTH classes in every batch
#   - CE on 50/50 batches matches the 50/50 balanced test set
balanced_train_sampler = BalancedBatchSampler(y_train, BATCH_SIZE)
balanced_train_loader  = DataLoader(train_dataset, batch_sampler=balanced_train_sampler,
                                    num_workers=2, pin_memory=True)

val_loader  = DataLoader(val_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches/epoch: {len(balanced_train_loader):,}  '
      f'({BATCH_SIZE//2} genuine + {BATCH_SIZE//2} fake per batch)')

In [ ]:
class DistilBertContrastive(nn.Module):
    """
    DistilBERT encoder with:
      - a projection head for SupCon (used in training only, discarded at inference)
      - a DistilBertForSequenceClassification-compatible classifier head
        (so weights can be exported to HuggingFace format for ONNX)
    """
    def __init__(self, model_name, proj_hidden=256, proj_out=128):
        super().__init__()
        self.encoder       = DistilBertModel.from_pretrained(model_name)
        hidden             = self.encoder.config.hidden_size  # 768

        # Projection head — SupCon geometry only
        self.projection    = nn.Sequential(
            nn.Linear(hidden, proj_hidden),
            nn.ReLU(),
            nn.Linear(proj_hidden, proj_out),
        )

        # Classifier head — mirrors DistilBertForSequenceClassification
        self.pre_classifier = nn.Linear(hidden, hidden)
        self.dropout        = nn.Dropout(0.1)
        self.classifier     = nn.Linear(hidden, 2)

    def get_cls(self, input_ids, attention_mask):
        return self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]

    def _classify_from_cls(self, cls):
        pooled = F.relu(self.pre_classifier(cls))
        pooled = self.dropout(pooled)
        return self.classifier(pooled)

    def forward_classify(self, input_ids, attention_mask):
        return self._classify_from_cls(self.get_cls(input_ids, attention_mask))

    def forward_joint(self, input_ids, attention_mask):
        """One encoder pass → (logits for CE, L2-normalised projection for SupCon)."""
        cls    = self.get_cls(input_ids, attention_mask)
        logits = self._classify_from_cls(cls)
        proj   = F.normalize(self.projection(cls), dim=1)   # unit sphere
        return logits, proj


model = DistilBertContrastive(MODEL_NAME, PROJ_HIDDEN, PROJ_OUT).to(DEVICE)
print(f'Model loaded. Total params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
for path in [
    'yelp_fake_distilbert_contrastive_model',
    'yelp_fake_distilbert_contrastive_onnx',
    'yelp_fake_distilbert_contrastive_threshold.json',
    'yelp_fake_distilbert_contrastive_metrics.csv',
    'yelp_fake_distilbert_contrastive_config.json',
    'yelp_fake_distilbert_contrastive_onnx.zip',
]:
    if os.path.isdir(path):
        shutil.rmtree(path); print(f'Removed dir : {path}')
    elif os.path.isfile(path):
        os.remove(path);     print(f'Removed file: {path}')
print('Clean.')

In [ ]:
# ── Helpers + loss for joint training ────────────────────────────

def get_probs_and_labels(model, loader):
    """P(fake) and labels over a loader, using the classifier head only."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model.forward_classify(
                batch['input_ids'].to(DEVICE),
                batch['attention_mask'].to(DEVICE),
            )
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(batch['labels'].numpy())
    return np.array(all_probs), np.array(all_labels)


def best_macro_f1_threshold(probs, labels):
    best_t, best_f = 0.5, 0.0
    for t in np.arange(0.10, 0.90, 0.01):
        f = f1_score(labels, (probs >= t).astype(int), average='macro', zero_division=0)
        if f > best_f:
            best_f, best_t = f, float(t)
    return best_t, best_f


def build_llrd_optimizer(model, base_lr, decay, weight_decay=0.01):
    """
    Layer-wise learning-rate decay (LLRD).
      - heads (projection / pre_classifier / classifier)  → base_lr
      - top DistilBERT layer (5)                          → base_lr * decay
      - ...                                                  ...
      - bottom layer (0)                                  → base_lr * decay^6
      - embeddings                                        → base_lr * decay^7
    Biases and LayerNorm weights get no weight decay (standard).
    """
    n_layers = model.encoder.config.num_hidden_layers  # 6 for DistilBERT

    def lr_for(name):
        if name.startswith('encoder.embeddings'):
            return base_lr * (decay ** (n_layers + 1))
        if 'encoder.transformer.layer.' in name:
            li = int(name.split('encoder.transformer.layer.')[1].split('.')[0])
            return base_lr * (decay ** (n_layers - li))
        return base_lr  # heads

    groups = []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        wd = 0.0 if ('bias' in name or 'norm' in name.lower()) else weight_decay
        groups.append({'params': [p], 'lr': lr_for(name), 'weight_decay': wd})
    return AdamW(groups)


n_genuine = sum(1 for v in y_train if v == 0)
n_fake    = sum(1 for v in y_train if v == 1)
total     = len(y_train)
print(f'Training distribution — genuine: {n_genuine:,} ({n_genuine/total:.1%})  '
      f'fake: {n_fake:,} ({n_fake/total:.1%})')
print('Balanced batches handle imbalance — no class weights applied to CE.')

# Unweighted CE: balanced batches give equal genuine/fake contribution per step
criterion_clf = nn.CrossEntropyLoss()

In [ ]:
print('=' * 60)
print('Joint SupCon + Cross-Entropy fine-tuning')
print(f'  loss = (1 - {SUPCON_LAMBDA})*CE + {SUPCON_LAMBDA}*SupCon   |   '
      f'τ={SUPCON_TEMP}  max_len={MAX_LEN}  batch={BATCH_SIZE}')
print(f'  LLRD base_lr={LR} decay={LLRD_DECAY}  |  cosine schedule, warmup={WARMUP_RATIO}')
print('=' * 60)

total_steps  = len(balanced_train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
optimizer = build_llrd_optimizer(model, base_lr=LR, decay=LLRD_DECAY, weight_decay=0.01)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps,
)

best_val_f1, best_thresh = 0.0, 0.5
patience_ctr   = 0
supcon_history = []   # per-epoch mean SupCon component
train_history  = []   # per-epoch {epoch, loss, ce, supcon, val_f1, threshold}

for epoch in range(1, EPOCHS + 1):
    model.train()
    ce_losses, sc_losses, tot_losses = [], [], []
    for step, batch in enumerate(balanced_train_loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)

        optimizer.zero_grad()
        logits, proj = model.forward_joint(input_ids, attention_mask)
        loss_ce = criterion_clf(logits, labels)
        loss_sc = supcon_loss(proj, labels, temperature=SUPCON_TEMP)
        loss    = (1.0 - SUPCON_LAMBDA) * loss_ce + SUPCON_LAMBDA * loss_sc
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        ce_losses.append(loss_ce.item())
        sc_losses.append(loss_sc.item())
        tot_losses.append(loss.item())

        if (step + 1) % 500 == 0:
            print(f'  Epoch {epoch} | Step {step+1}/{len(balanced_train_loader)} '
                  f'| CE: {np.mean(ce_losses):.4f} | SupCon: {np.mean(sc_losses):.4f} '
                  f'| Total: {np.mean(tot_losses):.4f}')

    mean_ce, mean_sc, mean_tot = np.mean(ce_losses), np.mean(sc_losses), np.mean(tot_losses)
    supcon_history.append(mean_sc)

    val_probs, val_labels = get_probs_and_labels(model, val_loader)
    val_thresh, val_f1    = best_macro_f1_threshold(val_probs, val_labels)
    train_history.append({'epoch': epoch, 'loss': mean_tot, 'ce': mean_ce,
                          'supcon': mean_sc, 'val_f1': val_f1, 'threshold': val_thresh})

    print(f'\nEpoch {epoch}/{EPOCHS} — CE: {mean_ce:.4f} | SupCon: {mean_sc:.4f} '
          f'| Total: {mean_tot:.4f} | Val Macro F1: {val_f1:.1%} | Threshold: {val_thresh:.2f}')

    if val_f1 > best_val_f1:
        best_val_f1, best_thresh, patience_ctr = val_f1, val_thresh, 0
        hf_model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
        hf_model.distilbert.load_state_dict(model.encoder.state_dict())
        hf_model.pre_classifier.weight.data = model.pre_classifier.weight.data.clone()
        hf_model.pre_classifier.bias.data   = model.pre_classifier.bias.data.clone()
        hf_model.classifier.weight.data     = model.classifier.weight.data.clone()
        hf_model.classifier.bias.data       = model.classifier.bias.data.clone()
        hf_model.save_pretrained('yelp_fake_distilbert_contrastive_model')
        tokenizer.save_pretrained('yelp_fake_distilbert_contrastive_model')
        print(f'  Best model saved (Val macro F1 = {best_val_f1:.1%}, threshold = {best_thresh:.2f})')
    else:
        patience_ctr += 1
        print(f'  No improvement ({patience_ctr}/{PATIENCE})')
        if patience_ctr >= PATIENCE:
            print(f'  Early stopping. Best Val macro F1: {best_val_f1:.1%}')
            break
    print()

print(f'\nTraining complete. Best Val macro F1: {best_val_f1:.1%} at threshold {best_thresh:.2f}')

In [ ]:
# ── Load best checkpoint for final evaluation ─────────────────────
from transformers import pipeline as hf_pipeline

best_hf = DistilBertForSequenceClassification.from_pretrained('yelp_fake_distilbert_contrastive_model')
best_hf.eval()

def get_probs_hf(hf_model, texts, batch_size=32):
    hf_model = hf_model.to(DEVICE)
    probs = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(
            texts[i:i+batch_size], max_length=MAX_LEN,
            padding='max_length', truncation=True, return_tensors='pt',
        ).to(DEVICE)
        with torch.no_grad():
            logits = hf_model(**enc).logits
        probs.extend(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
    return np.array(probs)

print('Running final threshold tuning on validation set ...')
val_probs_final = get_probs_hf(best_hf, X_val)
best_thresh_final, best_f1_final = best_macro_f1_threshold(val_probs_final, np.array(y_val))
print(f'Final threshold: {best_thresh_final:.2f}  (Val macro F1: {best_f1_final:.1%})')

print('\nInternal test evaluation (imbalanced test set):')
test_probs_final = get_probs_hf(best_hf, X_test)
y_pred = (test_probs_final >= best_thresh_final).astype(int)
print(classification_report(y_test, y_pred, target_names=['Genuine', 'Fake']))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ep = [h['epoch'] for h in train_history]

# Left — SupCon component per epoch
axes[0].plot(ep, [h['supcon'] for h in train_history], marker='o', color='tab:blue')
axes[0].set_title('SupCon Loss per Epoch'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('SupCon')

# Right — total/CE loss + val macro F1
axes[1].plot(ep, [h['loss'] for h in train_history], marker='o', label='Total Loss')
axes[1].plot(ep, [h['ce']   for h in train_history], marker='^', label='CE Loss')
ax2 = axes[1].twinx()
ax2.plot(ep, [h['val_f1'] for h in train_history], marker='s', color='orange', label='Val Macro F1')
axes[1].set_title('Loss & Val Macro F1'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
ax2.set_ylabel('Val Macro F1')
axes[1].legend(loc='upper left'); ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# ── Save threshold JSON ───────────────────────────────────────────
threshold_path = 'yelp_fake_distilbert_contrastive_threshold.json'
with open(threshold_path, 'w') as f:
    json.dump({'threshold': round(float(best_thresh_final), 2)}, f, indent=2)
print(f'Threshold saved: {threshold_path}  (value: {best_thresh_final:.2f})')

# ── Save metrics CSV ─────────────────────────────────────────────
y_pred_arr = np.array(y_pred)
y_true_arr = np.array(y_test)
metrics = {
    'model':                  'distilbert_contrastive',
    'threshold':              round(float(best_thresh_final), 2),
    'val_macro_f1':           round(best_f1_final, 4),
    'test_accuracy':          round(accuracy_score(y_true_arr, y_pred_arr), 4),
    'test_macro_f1':          round(f1_score(y_true_arr, y_pred_arr, average='macro'), 4),
    'test_fake_precision':    round(precision_score(y_true_arr, y_pred_arr, pos_label=1, zero_division=0), 4),
    'test_fake_recall':       round(recall_score(y_true_arr, y_pred_arr, pos_label=1, zero_division=0), 4),
    'test_fake_f1':           round(f1_score(y_true_arr, y_pred_arr, pos_label=1, zero_division=0), 4),
    'test_genuine_precision': round(precision_score(y_true_arr, y_pred_arr, pos_label=0, zero_division=0), 4),
    'test_genuine_recall':    round(recall_score(y_true_arr, y_pred_arr, pos_label=0, zero_division=0), 4),
    'test_genuine_f1':        round(f1_score(y_true_arr, y_pred_arr, pos_label=0, zero_division=0), 4),
}
pd.DataFrame([metrics]).to_csv('yelp_fake_distilbert_contrastive_metrics.csv', index=False)
print('Metrics saved: yelp_fake_distilbert_contrastive_metrics.csv')

# ── Save config JSON ─────────────────────────────────────────────
config = {
    'model_name': MODEL_NAME, 'max_len': MAX_LEN,
    'proj_hidden': PROJ_HIDDEN, 'proj_out': PROJ_OUT,
    'method': 'joint_supcon_ce',
    'supcon_temp': SUPCON_TEMP, 'supcon_lambda': SUPCON_LAMBDA,
    'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'lr': LR,
    'llrd_decay': LLRD_DECAY, 'schedule': 'cosine',
    'warmup_ratio': WARMUP_RATIO, 'patience': PATIENCE,
}
with open('yelp_fake_distilbert_contrastive_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print('Config saved: yelp_fake_distilbert_contrastive_config.json')

In [ ]:
print('Exporting to ONNX ...')

ort_model = ORTModelForSequenceClassification.from_pretrained(
    'yelp_fake_distilbert_contrastive_model', export=True
)
ort_model.save_pretrained('yelp_fake_distilbert_contrastive_onnx')
tokenizer.save_pretrained('yelp_fake_distilbert_contrastive_onnx')

# Copy threshold.json into the ONNX folder so evaluate_yelp_models.py can find it
import shutil
shutil.copy(threshold_path, 'yelp_fake_distilbert_contrastive_onnx/threshold.json')
print('ONNX model saved to: yelp_fake_distilbert_contrastive_onnx/')

In [ ]:
from transformers import AutoTokenizer, pipeline as hf_pipeline

ort_verify = ORTModelForSequenceClassification.from_pretrained(
    'yelp_fake_distilbert_contrastive_onnx', provider='CPUExecutionProvider'
)
tok_verify = AutoTokenizer.from_pretrained('yelp_fake_distilbert_contrastive_onnx')
pipe       = hf_pipeline('text-classification', model=ort_verify, tokenizer=tok_verify, device=-1)

samples = [
    'Great food and amazing service, highly recommend!',
    'Absolutely the best place ever!!! 5 stars!!!',
    'Terrible experience, cold food and rude staff.',
]
for text, result in zip(samples, pipe(samples)):
    label = 'FAKE' if result['label'] == 'LABEL_1' else 'GENUINE'
    print(f'{label} ({result["score"]:.1%}) — "{text[:60]}"')

In [ ]:
import zipfile
from google.colab import files

shutil.make_archive('yelp_fake_distilbert_contrastive_onnx', 'zip', 'yelp_fake_distilbert_contrastive_onnx')

# Also include metrics and config in the zip
with zipfile.ZipFile('yelp_fake_distilbert_contrastive_onnx.zip', 'a') as zf:
    zf.write('yelp_fake_distilbert_contrastive_metrics.csv')
    zf.write('yelp_fake_distilbert_contrastive_config.json')

print('Created: yelp_fake_distilbert_contrastive_onnx.zip')
files.download('yelp_fake_distilbert_contrastive_onnx.zip')

## Next steps
Extract `yelp_fake_distilbert_contrastive_onnx.zip` and place the folder at:
```
data/yelp_fake_distilbert_contrastive_onnx/
    model.onnx
    config.json
    tokenizer.json
    vocab.txt
    threshold.json
    yelp_fake_distilbert_contrastive_metrics.csv
    yelp_fake_distilbert_contrastive_config.json
    ...
```
Then run `evaluate_yelp_ablation.py` to compare all six models on the balanced test set.